# Notebook 28 — ERA5 Moisture + Low-Level Winds Download (MJO moisture-constraint experiment)
**Project:** ENSO-BSISO SSL — MJO extension · moisture-constraint experiment (Zhang et al. 2020 motivated)
**Author:** Jiayi (jh9141@nyu.edu)

Downloads the fields needed to diagnose the **moisture-convection phase relationship** of the MJO
(Zhang et al. 2020). Same domain/grid/period as the existing MJO pipeline (nb12), so everything aligns
with `X_MJO` and the RMM labels.

| Field | ERA5 variable | Why |
|---|---|---|
| **TCWV** (total column water vapour) | `total_column_water_vapour` (single-level) | column moisture -> **moisture-mode** test |
| **q** at 1000/925/850/700 hPa | `specific_humidity` (pressure-levels) | **lower-tropospheric** moisture (1000-700) -> **skeleton** test; gives dq/dt |
| **u,v** at 1000/925 hPa | `u/v_component_of_wind` (pressure-levels) | low-level **divergence** -> BL-convergence lead (**trio-interaction**) |

(Rossby-Kelvin ratio needs no new data — u850 is already channel 0 of `X_MJO`.)

**Domain:** 15S-15N, all longitudes, 2x2 deg. **Period:** 1979-2023, daily mean (4x/day -> mean).
**Each year is split into two half-year sub-requests** (months 1-6 and 7-12) to stay under the CDS
cost limit — same trick nb12 used for OLR.
**Output (`BSISO_SSL_Project/MJO/moisture_constraints/data/raw/`):**
```
TCWV_{yr}.nc          qplev_{yr}.nc          uvplev_low_{yr}.nc      (45 annual files each)
```

---

## Cell 1 — Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_DIR     = f'{PROJECT_DIR}/MJO'
MOIST_RAW   = f'{MJO_DIR}/moisture_constraints/data/raw'
os.makedirs(MOIST_RAW, exist_ok=True)
print('Raw moisture dir:', MOIST_RAW)
for f in sorted(os.listdir(MOIST_RAW)):
    print('  ', f, round(os.path.getsize(f'{MOIST_RAW}/{f}')/1e6, 1), 'MB')

## Cell 2 — Install CDS API Client

In [ ]:
get_ipython().system('pip install cdsapi --quiet')
import cdsapi
print('cdsapi ready.')

## Cell 3 — CDS API Credentials
Same personal access token as the BSISO/MJO project.

In [ ]:
# ============================================================
# FILL IN YOUR PERSONAL ACCESS TOKEN HERE
# https://cds.climate.copernicus.eu/  ->  Your profile
# ============================================================
CDS_API_KEY = 'YOUR_CDS_API_KEY_HERE'
# ============================================================
import os
with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write(f'url: https://cds.climate.copernicus.eu/api\nkey: {CDS_API_KEY}\n')
print('CDS credentials saved.')
try:
    import cdsapi; cdsapi.Client(quiet=True); print('CDS API connection: OK')
except Exception as e:
    print('CDS API connection FAILED:', e)

## Cell 4 — Shared config + half-year-split daily downloader

All fields are **instantaneous** -> 4x/day (00/06/12/18 UTC) averaged to a daily mean. Each year is
fetched as **two half-year sub-requests** (months 1-6, 7-12), each aggregated to daily and concatenated,
so no single request exceeds the CDS cost limit. `pressure_level` is preserved when present.

In [ ]:
import cdsapi, os
import xarray as xr

if not os.path.exists(os.path.expanduser('~/.cdsapirc')):
    raise RuntimeError('Run Cell 3 first to set up CDS credentials.')
client = cdsapi.Client()

DAYS         = [f'{d:02d}' for d in range(1, 32)]      # CDS ignores invalid dates
INST_TIMES   = ['00:00', '06:00', '12:00', '18:00']    # instantaneous -> daily MEAN
AREA         = [15, -180, -15, 180]                    # N, W, S, E (global 15S-15N)
GRID         = [2.0, 2.0]
YEARS        = list(range(1979, 2024))
MONTH_HALVES = [['01','02','03','04','05','06'], ['07','08','09','10','11','12']]

# CRITICAL: temp 4x/day sub-files go to LOCAL Colab disk, NOT Drive.
# On Google Drive, os.remove() moves files to TRASH, which keeps counting against your
# 15 GB quota for 30 days -> the huge temp sub-files were what filled the Drive mid-download.
# Writing them to /content/ (ephemeral, ~100 GB) makes os.remove a real delete. Only the small
# final daily file is written to Drive.
LOCAL_TMP = '/content/_era5_tmp'
os.makedirs(LOCAL_TMP, exist_ok=True)

def download_year_daily(collection, base_req, out, keep_vars):
    # Download one year as two half-year requests (LOCAL temp), aggregate to daily, concat -> Drive.
    if os.path.exists(out):
        print(f'[SKIP] {os.path.basename(out)} ({os.path.getsize(out)/1e6:.1f} MB)'); return
    parts = []
    for h, months in enumerate(MONTH_HALVES):
        sub = f'{LOCAL_TMP}/' + os.path.basename(out).replace('.nc', f'_h{h}.nc')   # LOCAL temp
        req = dict(base_req); req.update({'product_type':'reanalysis','month':months,'day':DAYS,
                                          'time':INST_TIMES,'area':AREA,'grid':GRID,'data_format':'netcdf'})
        print(f'  {os.path.basename(out)} months {months[0]}-{months[-1]} ...', end=' ', flush=True)
        client.retrieve(collection, req, sub)
        ds = xr.open_dataset(sub); tdim = 'valid_time' if 'valid_time' in ds.dims else 'time'
        daily = ds[keep_vars].reset_coords(drop=True).resample(**{tdim:'1D'}).mean().dropna(dim=tdim, how='all').load()
        parts.append(daily); ds.close()
        os.remove(sub)                                  # local disk -> real delete, no Drive Trash
        print('done')
    tdim = 'valid_time' if 'valid_time' in parts[0].dims else 'time'
    xr.concat(parts, dim=tdim).sortby(tdim).to_netcdf(out)              # only small final file -> Drive
    print(f'  saved {os.path.basename(out)} ({os.path.getsize(out)/1e6:.1f} MB)')

print('Config ready. Temp subs ->', LOCAL_TMP, '(local, not Drive). Final files -> Drive.')

## Cell 5 — Download TCWV (column moisture)

In [ ]:
print(f'{len(YEARS)} annual TCWV chunks. Existing files skipped.')
for yr in YEARS:
    download_year_daily('reanalysis-era5-single-levels',
        {'variable':'total_column_water_vapour', 'year':str(yr)},
        f'{MOIST_RAW}/TCWV_{yr}.nc', keep_vars=['tcwv'])
print('\nTCWV done.')

## Cell 6 — Download specific humidity q at 1000/925/850/700 hPa
For the lower-tropospheric (1000-700 hPa) moisture integral, computed in nb29.

In [ ]:
Q_LEVELS = ['1000', '925', '850', '700']
print(f'{len(YEARS)} annual q chunks at levels {Q_LEVELS}. Existing files skipped.')
for yr in YEARS:
    download_year_daily('reanalysis-era5-pressure-levels',
        {'variable':'specific_humidity', 'pressure_level':Q_LEVELS, 'year':str(yr)},
        f'{MOIST_RAW}/qplev_{yr}.nc', keep_vars=['q'])
print('\nq done.')

## Cell 7 — Download low-level winds u,v at 1000/925 hPa
For low-level divergence (BL convergence lead, trio-interaction).

In [ ]:
UV_LEVELS = ['1000', '925']
print(f'{len(YEARS)} annual u,v chunks at levels {UV_LEVELS}. Existing files skipped.')
for yr in YEARS:
    download_year_daily('reanalysis-era5-pressure-levels',
        {'variable':['u_component_of_wind','v_component_of_wind'], 'pressure_level':UV_LEVELS, 'year':str(yr)},
        f'{MOIST_RAW}/uvplev_low_{yr}.nc', keep_vars=['u','v'])
print('\nu,v done.')

## Cell 8 — Verify downloads

In [ ]:
import os, xarray as xr, numpy as np, pandas as pd
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
MOIST_RAW = '/content/drive/MyDrive/BSISO_SSL_Project/MJO/moisture_constraints/data/raw'

def check(prefix, var, levels=False):
    fs = sorted(f'{MOIST_RAW}/{f}' for f in os.listdir(MOIST_RAW)
                if f.startswith(prefix) and f.endswith('.nc'))
    print(f'\n[{prefix}] {len(fs)}/45 annual files')
    if not fs: return
    ds = xr.concat([xr.open_dataset(f) for f in fs], dim='valid_time').sortby('valid_time')
    t = pd.DatetimeIndex(ds.valid_time.values)
    print(f'  days {len(t)}  {t[0].date()}..{t[-1].date()}  months {sorted(set(t.month))}')
    print(f'  grid {len(ds.latitude)} lat x {len(ds.longitude)} lon', end='')
    if levels: print(f'  levels {sorted(ds.pressure_level.values.tolist())}')
    else: print()
    v = ds[var].values
    print(f'  {var} range [{np.nanmin(v):.3g}, {np.nanmax(v):.3g}]  NaN {int(np.isnan(v).sum())}')
    yrs = sorted(set(t.year)); miss = [y for y in range(1979,2024) if y not in yrs]
    if miss: print(f'  MISSING years: {miss}')
    ds.close()

check('TCWV_', 'tcwv')
check('qplev_', 'q', levels=True)
check('uvplev_low_', 'u', levels=True)
print('\nVerification complete. Next: nb29 (preprocess).')

---
## Done!
`MJO/moisture_constraints/data/raw/` now holds `TCWV_*.nc`, `qplev_*.nc`, `uvplev_low_*.nc` (45 each).

**Next:** `29_mjo_moisture_preprocess.ipynb`.

---
*DDCS Project | jh9141@nyu.edu*